# CelebA-HQ — standalone validation split (Stage 3 only)

Use this small notebook to continue from saved Stage 2 artifacts without rerunning the path scan, SHA-256, or image decode stages. Attach a Kaggle Dataset containing the saved Stage 2 summary JSON and JSONL manifest as an input; SQLite is optional. This notebook is preconfigured for the attached artifact Dataset at `/kaggle/input/datasets/j4cknguyn/dire-celeba-hq-stage-2-artifacts-v1`. If Kaggle mounts it at another path, update `ARTIFACT_DIR` in the runner cell. You do **not** need to attach the 40+ GiB DIRE image Dataset here.

This notebook reads only the summary and JSONL from the attached read-only artifact input, performs no image/GPU work, and writes the candidate split JSON only to `/kaggle/working`. The summary's embedded manifest path may point to an old runtime; the explicit mounted JSONL path is passed as an override, so do not edit the saved summary.

The content-disjoint v2 rule excludes any record whose SHA-256 occurs more than once across the training pool, then fills each exact validation quota using the next locator-rank candidates. Every copy from repeated-hash groups remains in training. After running, review the exact counts, candidate audit, validation ID-list SHA-256, and content-overlap status; `review_required` means stop before training.


In [ ]:
"""Create the approved deterministic CelebA-HQ train/validation ID split.

The splitter only reads the integrity summary/JSONL produced by Stage 2 and
writes a candidate split manifest beneath a caller-provided directory outside
the mounted input. It never opens images or changes the data tree.
"""

from __future__ import annotations

import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

VALIDATION_SALT = "dire-celebahq-val-v1"
EXPECTED_TRAIN_COUNTS = {("real", 0): 28_000, ("sdv2", 1): 40_000}
EXPECTED_TOTAL_RECORDS = 72_600


def _read_jsonl(path: Path) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    with path.open("r", encoding="utf-8") as stream:
        try:
            metadata = json.loads(next(stream))
        except StopIteration as exc:
            raise ValueError("integrity JSONL manifest is empty") from exc
        rows = [json.loads(line) for line in stream if line.strip()]
    if metadata.get("record_type") != "metadata":
        raise ValueError("integrity JSONL first line is not metadata")
    return metadata, rows


def _round_half_up_ratio(n: int, numerator: int, denominator: int) -> int:
    """Round ``n * numerator / denominator`` to nearest integer, ties upward."""
    return (2 * n * numerator + denominator) // (2 * denominator)


def _rank_digest(salt: str, locator: str) -> str:
    return hashlib.sha256(salt.encode("utf-8") + b"\0" + locator.encode("utf-8")).hexdigest()


def _canonical_list_sha256(values: list[Any]) -> str:
    payload = json.dumps(values, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def create_celebahq_validation_split(
    integrity_summary_path: str | Path,
    output_dir: str | Path,
    *,
    integrity_manifest_path: str | Path | None = None,
    salt: str = VALIDATION_SALT,
    expected_train_counts: dict[tuple[str, int], int] | None = None,
    expected_total_records: int | None = EXPECTED_TOTAL_RECORDS,
) -> dict[str, Any]:
    """Select exactly 10% of each training label using the locked hash-rank rule.

    The integrity summary and its JSONL manifest must describe the same complete,
    failure-free Stage 2 run. ``integrity_manifest_path`` can override the path
    embedded in the summary, allowing artifacts mounted read-only from a separate
    Kaggle Dataset. Only ``train/real`` and ``train/sdv2`` are split; all test
    records are excluded. All train rows sharing a SHA-256 with any other train
    row are ineligible for validation, globally across both labels; validation
    fills the exact per-label quota from the next unique-content rank candidates.
    """
    summary_path = Path(integrity_summary_path).expanduser().resolve()
    if not summary_path.is_file():
        raise FileNotFoundError(f"integrity summary not found: {summary_path}")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if summary.get("status") != "pass":
        raise ValueError("integrity summary status is not pass; refusing to create validation IDs")
    if summary.get("input_modified") is not False or summary.get("gpu_work_started") is not False:
        raise ValueError("integrity summary does not attest to read-only, no-GPU Stage 2")
    for key in ("hash_failures", "missing_hashes", "decode_failures"):
        if summary.get(key) != 0:
            raise ValueError(f"integrity summary has {key}={summary.get(key)!r}; refusing split")

    record_count = summary.get("records_in_checkpoint")
    if not isinstance(record_count, int) or record_count < 1:
        raise ValueError("integrity summary has an invalid records_in_checkpoint")
    if summary.get("records_expected") != record_count:
        raise ValueError("integrity summary expected/checkpoint record counts differ")
    if expected_total_records is not None and record_count != expected_total_records:
        raise ValueError(
            f"expected {expected_total_records} integrity records, got {record_count}"
        )

    manifest_value = integrity_manifest_path or summary.get("file_manifest_jsonl")
    if not isinstance(manifest_value, (str, Path)) or not str(manifest_value):
        raise ValueError("integrity summary does not name file_manifest_jsonl")
    manifest_path = Path(manifest_value).expanduser().resolve()
    if not manifest_path.is_file():
        raise FileNotFoundError(f"integrity JSONL manifest not found: {manifest_path}")
    manifest_meta, records = _read_jsonl(manifest_path)
    if len(records) != record_count or manifest_meta.get("records") != record_count:
        raise ValueError("integrity JSONL record count does not match its summary")
    if manifest_meta.get("source_fingerprint") != summary.get("source_fingerprint"):
        raise ValueError("integrity JSONL and summary source fingerprints differ")
    if manifest_meta.get("archive_member_paths_verified") is not False:
        raise ValueError("unexpected archive-member path verification claim")

    seen_paths: set[str] = set()
    classes: dict[int, list[dict[str, Any]]] = defaultdict(list)
    observed_train_counts: Counter[tuple[str, int]] = Counter()
    for record in records:
        if record.get("hash_status") != "pass" or record.get("decode_status") != "pass":
            raise ValueError(f"integrity failure in manifest row: {record.get('path_from_dire_root')}")
        image_hash = record.get("sha256")
        if not isinstance(image_hash, str) or len(image_hash) != 64:
            raise ValueError(f"missing/invalid SHA-256 for {record.get('path_from_dire_root')}")
        try:
            int(image_hash, 16)
        except ValueError as exc:
            raise ValueError(f"invalid SHA-256 for {record.get('path_from_dire_root')}") from exc
        path_id = record.get("path_from_dire_root")
        if not isinstance(path_id, str) or not path_id or path_id in seen_paths:
            raise ValueError(f"missing or duplicate root-relative path: {path_id!r}")
        seen_paths.add(path_id)

        if record.get("split") != "train":
            continue
        generator = record.get("generator")
        label = record.get("label")
        if (generator, label) not in EXPECTED_TRAIN_COUNTS:
            raise ValueError(f"unexpected CelebA-HQ training group: {(generator, label)!r}")
        locator = record.get("member_locator_inferred")
        if not isinstance(locator, str) or not locator:
            raise ValueError(f"missing inferred member locator for {path_id}")
        if record.get("locator_inferred") is not True:
            raise ValueError(f"expected locator_inferred=true for {path_id}")
        observed_train_counts[(generator, label)] += 1
        classes[label].append(record)

    required_counts = expected_train_counts or EXPECTED_TRAIN_COUNTS
    if dict(observed_train_counts) != required_counts:
        raise ValueError(
            f"training group counts differ from locked inventory: "
            f"observed={dict(observed_train_counts)}, expected={required_counts}"
        )
    if set(classes) != {0, 1}:
        raise ValueError(f"expected binary train labels 0 and 1, got {sorted(classes)}")
    for label, class_records in classes.items():
        locators = [row["member_locator_inferred"] for row in class_records]
        if len(locators) != len(set(locators)):
            raise ValueError(f"duplicate inferred member locator within label {label}")

    all_train_records = [record for class_records in classes.values() for record in class_records]
    global_sha_counts = Counter(record["sha256"] for record in all_train_records)
    validation_records: list[dict[str, Any]] = []
    class_counts: dict[str, dict[str, int]] = {}
    candidate_audit_by_label: dict[str, dict[str, int]] = {}
    validation_path_ids: set[str] = set()
    for label in sorted(classes):
        ranked = sorted(
            classes[label],
            key=lambda row: (
                _rank_digest(salt, row["member_locator_inferred"]),
                row["member_locator_inferred"],
            ),
        )
        selected_count = _round_half_up_ratio(len(ranked), 1, 10)
        eligible = [
            (rank_position, row)
            for rank_position, row in enumerate(ranked, start=1)
            if global_sha_counts[row["sha256"]] == 1
        ]
        excluded_duplicate_candidates = len(ranked) - len(eligible)
        if len(eligible) < selected_count:
            raise ValueError(
                f"not enough content-unique candidates for label {label}: "
                f"need {selected_count}, have {len(eligible)}"
            )
        chosen = eligible[:selected_count]
        for validation_rank, (rank_position, row) in enumerate(chosen, start=1):
            path_id = row["path_from_dire_root"]
            validation_path_ids.add(path_id)
            validation_records.append(
                {
                    "path_from_dire_root": path_id,
                    "generator": row["generator"],
                    "label": label,
                    "member_locator_inferred": row["member_locator_inferred"],
                    "locator_inferred": True,
                    "sha256": row["sha256"],
                    "validation_rank": validation_rank,
                    "rank_position_before_duplicate_filter": rank_position,
                    "selection_sha256": _rank_digest(salt, row["member_locator_inferred"]),
                }
            )
        class_counts[str(label)] = {
            "total": len(ranked),
            "validation": selected_count,
            "training": len(ranked) - selected_count,
        }
        candidate_audit_by_label[str(label)] = {
            "total_records": len(ranked),
            "content_unique_candidates": len(eligible),
            "excluded_duplicate_hash_candidates": excluded_duplicate_candidates,
        }

    validation_records.sort(key=lambda row: (row["label"], row["validation_rank"]))
    validation_id_list = sorted(
        [
            {
                "label": row["label"],
                "path_from_dire_root": row["path_from_dire_root"],
                "member_locator_inferred": row["member_locator_inferred"],
            }
            for row in validation_records
        ],
        key=lambda item: (item["label"], item["member_locator_inferred"], item["path_from_dire_root"]),
    )
    validation_hashes_all = {
        row["sha256"] for row in all_train_records
        if row["path_from_dire_root"] in validation_path_ids
    }
    training_hashes_all = {
        row["sha256"] for row in all_train_records
        if row["path_from_dire_root"] not in validation_path_ids
    }
    shared_hashes_all = sorted(validation_hashes_all & training_hashes_all)
    shared_hashes: dict[str, list[str]] = {}
    for label in sorted(classes):
        validation_hashes = {
            row["sha256"] for row in classes[label]
            if row["path_from_dire_root"] in validation_path_ids
        }
        training_hashes = {
            row["sha256"] for row in classes[label]
            if row["path_from_dire_root"] not in validation_path_ids
        }
        shared_hashes[str(label)] = sorted(validation_hashes & training_hashes)

    content_overlap = {
        "status": "review_required" if shared_hashes_all else "pass",
        "shared_sha256_count_all_labels": len(shared_hashes_all),
        "shared_sha256_all_labels": shared_hashes_all,
        "shared_sha256_count_by_label": {label: len(values) for label, values in shared_hashes.items()},
        "shared_sha256_by_label": shared_hashes,
        "note": "Exact duplicate bytes crossing train/validation are reported across both labels; the locked rank selection is not silently changed.",
    }
    output_path = Path(output_dir).expanduser().resolve()
    root_path = Path(summary.get("root", "")).expanduser().resolve()
    if not summary.get("root") or output_path == root_path or root_path in output_path.parents:
        raise ValueError("validation manifest output must be outside the read-only input root")
    output_path.mkdir(parents=True, exist_ok=True)
    validation_ids_digest = _canonical_list_sha256(validation_id_list)
    artifact = {
        "schema_version": 2,
        "status": "candidate_created",
        "protocol": {
            "name": "celebahq-content-disjoint-exact-count-hash-rank-v2",
            "salt": salt,
            "hash": "SHA-256",
            "preimage": "UTF-8(salt) + 0x00 + UTF-8(member_locator_inferred)",
            "sort": "ascending lowercase hex digest, then member_locator_inferred",
            "candidate_rule": "candidate SHA-256 frequency must equal 1 across all train labels; repeated-content groups remain wholly in training",
            "fraction": {"numerator": 1, "denominator": 10},
            "rounding": "half-up of total records per label before duplicate filtering",
        },
        "source": {
            "root": summary["root"],
            "source_fingerprint": summary["source_fingerprint"],
            "integrity_summary": str(summary_path),
            "integrity_manifest": str(manifest_path),
            "integrity_status": summary["status"],
            "records": record_count,
            "archive_member_paths_verified": False,
        },
        "counts_by_label": class_counts,
        "candidate_audit_by_label": candidate_audit_by_label,
        "duplicate_sha256_groups_all_train_labels": sum(
            1 for frequency in global_sha_counts.values() if frequency > 1
        ),
        "validation_id_list_sha256": validation_ids_digest,
        "content_overlap_audit": content_overlap,
        "validation_records": validation_records,
        "notes": [
            "Only train/real and train/sdv2 records are partitioned; all test records remain untouched.",
            "Repeated SHA-256 groups across either training label are excluded from validation candidates and remain entirely in training.",
            "Locators are inferred from the extracted tree and have not been verified against original tar member paths.",
            "Candidate split creation does not authorize training; review content_overlap_audit and complete the compute preflight first.",
        ],
    }
    destination = output_path / f"celebahq-validation-split-v2-{summary['source_fingerprint'][:12]}.json"
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_text(json.dumps(artifact, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, destination)
    artifact["artifact_path"] = str(destination)
    return artifact


In [ ]:
# Exact mount path reported for the attached Stage 2 artifact Dataset.
from pathlib import Path

ARTIFACT_DIR = Path(
    "/kaggle/input/datasets/j4cknguyn/dire-celeba-hq-stage-2-artifacts-v1"
)
if not ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(
        f"Artifact Dataset input not found at {ARTIFACT_DIR}. "
        f"Visible inputs: {[str(p) for p in Path('/kaggle/input').iterdir()]}"
    )
SUMMARY_NAME = "celebahq-integrity-81dcda99aeb997b25b82-summary.json"
MANIFEST_NAME = "celebahq-integrity-81dcda99aeb997b25b82.jsonl"


def _find_artifact(name: str) -> Path:
    direct = ARTIFACT_DIR / name
    if direct.is_file():
        return direct
    matches = list(ARTIFACT_DIR.rglob(name))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {name} under {ARTIFACT_DIR}, found {len(matches)}. "
            "Check ARTIFACT_DIR and the attached artifact Dataset."
        )
    return matches[0]


_integrity_summary_path = _find_artifact(SUMMARY_NAME)
_integrity_manifest_path = _find_artifact(MANIFEST_NAME)
validation_result = create_celebahq_validation_split(
    integrity_summary_path=_integrity_summary_path,
    integrity_manifest_path=_integrity_manifest_path,
    output_dir=Path("/kaggle/working"),
)
print("Integrity summary input:", _integrity_summary_path)
print("Integrity JSONL input:", _integrity_manifest_path)
print("Validation candidate:", validation_result["status"])
print("Counts by label:", validation_result["counts_by_label"])
print("Candidate audit by label:", validation_result["candidate_audit_by_label"])
print("Validation ID-list SHA-256:", validation_result["validation_id_list_sha256"])
print("Content-overlap audit:", validation_result["content_overlap_audit"]["status"])
print("Split artifact:", validation_result["artifact_path"])


## Stage 4 — Kaggle resource preflight (no training)

Run after Stage 3 v2 passes. Attach both the existing DIRE input Dataset and the Stage 2 artifact Dataset. This cell checks only required directories/files, current `/kaggle/working` disk capacity, tiny write/delete probes for planned artifact/checkpoint paths, and GPU metadata using `nvidia-smi` with a PyTorch device-properties fallback if the executable is missing. It does not open images, write into any input, allocate a CUDA tensor, run a GPU workload, or train. To read the assigned GPU details, select a Kaggle GPU accelerator; the query itself does no model compute.

The preflight writes `celebahq-training-preflight.json` under `/kaggle/working`. A `needs_attention` result blocks budget planning until missing paths/write access/GPU assignment are addressed. This report is only an environment snapshot; it does not select seed/budget or authorize training.


In [ ]:
"""Read-only Kaggle resource preflight for the CelebA-HQ training gate.

This performs metadata/path checks, tiny write/delete probes outside the mounted
inputs, and a metadata-only GPU query via ``nvidia-smi`` with a PyTorch device
property fallback. It does not open images, allocate tensors, run a GPU workload,
or start training.
"""

from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

EXPECTED_DIRE_ROOT = Path("/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire")
EXPECTED_ARTIFACT_INPUT = Path(
    "/kaggle/input/datasets/j4cknguyn/dire-celeba-hq-stage-2-artifacts-v1"
)
EXPECTED_SPLIT_COUNTS = {
    "0": {"total": 28_000, "validation": 2_800, "training": 25_200},
    "1": {"total": 40_000, "validation": 4_000, "training": 36_000},
}
EXPECTED_SPLIT_PROTOCOL = "celebahq-content-disjoint-exact-count-hash-rank-v2"


def _inside(path: Path, root: Path) -> bool:
    return path == root or root in path.parents


def _probe_writable(directory: Path) -> dict[str, Any]:
    try:
        directory.mkdir(parents=True, exist_ok=True)
        fd, name = tempfile.mkstemp(prefix=".celebahq-preflight-", dir=directory)
        probe = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(b"dire-celebahq-preflight\n")
                stream.flush()
                os.fsync(stream.fileno())
        finally:
            probe.unlink(missing_ok=True)
        return {"status": "pass", "path": str(directory), "probe_bytes": 24, "probe_removed": True}
    except OSError as exc:
        return {"status": "fail", "path": str(directory), "error": f"{type(exc).__name__}: {exc}"}


def _torch_gpu_fallback(nvidia_smi_error: dict[str, Any]) -> dict[str, Any]:
    """Query CUDA device metadata through PyTorch without allocating tensors."""
    try:
        import torch
    except Exception as exc:  # PyTorch may be absent or fail to import on CPU-only images.
        return {
            "status": "unavailable",
            "fallback": "torch.cuda metadata",
            "nvidia_smi": nvidia_smi_error,
            "error": f"{type(exc).__name__}: {exc}",
            "devices": [],
        }
    try:
        if not torch.cuda.is_available():
            return {
                "status": "no_gpu_visible",
                "fallback": "torch.cuda metadata",
                "nvidia_smi": nvidia_smi_error,
                "devices": [],
                "note": "No CUDA tensor/model was allocated; device discovery found no visible GPU.",
            }
        devices = []
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            try:
                free_bytes, total_bytes = torch.cuda.mem_get_info(index)
            except Exception:
                free_bytes = total_bytes = None
            devices.append(
                {
                    "index": index,
                    "name": properties.name,
                    "uuid": getattr(properties, "uuid", None),
                    "memory_total_bytes": total_bytes or getattr(properties, "total_memory", None),
                    "memory_free_bytes": free_bytes,
                    "cuda_runtime_version": getattr(torch.version, "cuda", None),
                }
            )
        return {
            "status": "pass" if devices else "no_gpu_visible",
            "fallback": "torch.cuda metadata",
            "nvidia_smi": nvidia_smi_error,
            "devices": devices,
            "note": "Metadata query only; no tensor/model allocation or GPU computation was started.",
        }
    except Exception as exc:
        return {
            "status": "unavailable",
            "fallback": "torch.cuda metadata",
            "nvidia_smi": nvidia_smi_error,
            "error": f"{type(exc).__name__}: {exc}",
            "devices": [],
        }


def _query_gpu(nvidia_smi: str = "nvidia-smi", timeout_seconds: int = 10) -> dict[str, Any]:
    command = [
        nvidia_smi,
        "--query-gpu=name,uuid,memory.total,memory.free,driver_version",
        "--format=csv,noheader,nounits",
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=timeout_seconds, check=False)
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError) as exc:
        error = {"command": command, "error": f"{type(exc).__name__}: {exc}"}
        return _torch_gpu_fallback(error)
    if result.returncode != 0:
        error = {"command": command, "returncode": result.returncode, "stderr": result.stderr.strip()[:1000]}
        return _torch_gpu_fallback(error)

    devices = []
    for row in csv.reader(result.stdout.splitlines(), skipinitialspace=True):
        values = [value.strip() for value in row]
        if not values or not any(values):
            continue
        if len(values) != 5:
            error = {
                "command": command,
                "error": f"unexpected nvidia-smi row with {len(values)} fields",
                "raw_output": result.stdout[:2000],
            }
            return _torch_gpu_fallback(error)
        devices.append(
            {
                "name": values[0],
                "uuid": values[1],
                "memory_total_mib": values[2],
                "memory_free_mib": values[3],
                "driver_version": values[4],
            }
        )
    return {
        "status": "pass" if devices else "no_gpu_visible",
        "command": command,
        "devices": devices,
        "note": "Metadata query only; no GPU workload was started.",
    }


def _find_artifact(directory: Path, filename: str) -> Path | None:
    direct = directory / filename
    if direct.is_file():
        return direct.resolve()
    if not directory.is_dir():
        return None
    matches = list(directory.rglob(filename))
    return matches[0].resolve() if len(matches) == 1 else None


def _inspect_split(path: Path, dire_root: Path) -> dict[str, Any]:
    if not path.is_file():
        return {"status": "fail", "path": str(path), "error": "split artifact not found"}
    try:
        split = json.loads(path.read_text(encoding="utf-8"))
        source_root = Path(split["source"]["root"]).expanduser().resolve()
        valid = (
            split.get("status") == "candidate_created"
            and split.get("protocol", {}).get("name") == EXPECTED_SPLIT_PROTOCOL
            and split.get("content_overlap_audit", {}).get("status") == "pass"
            and split.get("counts_by_label") == EXPECTED_SPLIT_COUNTS
            and source_root == dire_root.resolve()
        )
        return {
            "status": "pass" if valid else "fail",
            "path": str(path.resolve()),
            "protocol": split.get("protocol", {}).get("name"),
            "validation_id_list_sha256": split.get("validation_id_list_sha256"),
            "counts_by_label": split.get("counts_by_label"),
            "content_overlap_status": split.get("content_overlap_audit", {}).get("status"),
            "source_root_matches_config": source_root == dire_root.resolve(),
            "error": None if valid else "split metadata/counts/source/audit do not match the accepted v2 gate",
        }
    except (OSError, json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
        return {"status": "fail", "path": str(path), "error": f"{type(exc).__name__}: {exc}"}


def run_celebahq_training_preflight(
    *,
    dire_root: str | Path,
    stage2_artifact_dir: str | Path,
    split_manifest_path: str | Path,
    working_dir: str | Path,
    artifact_dir: str | Path,
    checkpoint_dir: str | Path,
    report_path: str | Path | None = None,
    nvidia_smi: str = "nvidia-smi",
) -> dict[str, Any]:
    """Collect a no-training resource/read-write preflight and save its report.

    The only writes are small, temporary probe files and the JSON report beneath
    ``working_dir``. Mounted input trees are inspected by path existence only and
    are never opened for writing or traversed recursively (other than the small
    Stage 2 artifact directory when looking up its two filenames).
    """
    work = Path(working_dir).expanduser().resolve()
    outputs = {
        "artifact_dir": Path(artifact_dir).expanduser().resolve(),
        "checkpoint_dir": Path(checkpoint_dir).expanduser().resolve(),
    }
    report = (
        Path(report_path).expanduser().resolve()
        if report_path is not None
        else work / "celebahq-training-preflight.json"
    )
    input_mount = Path("/kaggle/input").resolve()
    for destination in (work, *outputs.values(), report):
        if _inside(destination, input_mount):
            raise ValueError(f"preflight output must not be written beneath Kaggle input: {destination}")
        if not _inside(destination, work):
            raise ValueError(f"preflight outputs must remain beneath working_dir: {destination}")

    dire = Path(dire_root).expanduser().resolve()
    stage2_dir = Path(stage2_artifact_dir).expanduser().resolve()
    input_checks = {
        "dire_root_exists": dire.is_dir(),
        "train_real_group_exists": (dire / "train/celebahq/real/real").is_dir(),
        "train_sdv2_group_exists": (dire / "train/celebahq/sdv2/sdv2").is_dir(),
        "stage2_artifact_dataset_exists": stage2_dir.is_dir(),
    }
    summary_file = _find_artifact(
        stage2_dir, "celebahq-integrity-81dcda99aeb997b25b82-summary.json"
    )
    manifest_file = _find_artifact(
        stage2_dir, "celebahq-integrity-81dcda99aeb997b25b82.jsonl"
    )
    input_checks["stage2_summary_exists"] = summary_file is not None
    input_checks["stage2_jsonl_exists"] = manifest_file is not None

    split_check = _inspect_split(Path(split_manifest_path).expanduser().resolve(), dire)
    write_probes = {name: _probe_writable(path) for name, path in outputs.items()}
    try:
        usage = shutil.disk_usage(work)
        disk = {
            "status": "pass",
            "path": str(work),
            "total_bytes": usage.total,
            "used_bytes": usage.used,
            "free_bytes": usage.free,
            "note": "Reports current capacity only; sufficiency depends on the later approved training/checkpoint budget.",
        }
    except OSError as exc:
        disk = {"status": "fail", "path": str(work), "error": f"{type(exc).__name__}: {exc}"}

    gpu = _query_gpu(nvidia_smi=nvidia_smi)
    checks_passed = all(input_checks.values()) and split_check["status"] == "pass"
    checks_passed = checks_passed and all(probe["status"] == "pass" for probe in write_probes.values())
    checks_passed = checks_passed and disk["status"] == "pass" and gpu["status"] == "pass"
    result: dict[str, Any] = {
        "schema_version": 1,
        "status": "pass" if checks_passed else "needs_attention",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "scope": "Kaggle resource/path preflight only; no model compute",
        "inputs": {
            "dire_root": str(dire),
            "stage2_artifact_dir": str(stage2_dir),
            "stage2_summary": str(summary_file) if summary_file else None,
            "stage2_jsonl": str(manifest_file) if manifest_file else None,
        },
        "input_checks": input_checks,
        "split_check": split_check,
        "disk": disk,
        "write_probes": write_probes,
        "gpu_query": gpu,
        "execution_flags": {
            "image_contents_read": False,
            "input_write_attempted": False,
            "gpu_workload_started": False,
            "training_started": False,
        },
        "next_gate": "review this report, then decide seed/budget/checkpoint policy; this report does not authorize training",
        "report_path": str(report),
    }
    work.mkdir(parents=True, exist_ok=True)
    temporary = report.with_suffix(report.suffix + ".tmp")
    temporary.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, report)
    return result


In [ ]:
# Stage 4 runner — only after Stage 3 v2 candidate is present and overlap audit passed.
from pathlib import Path

if isinstance(globals().get("validation_result"), dict) and validation_result.get("artifact_path"):
    _split_manifest_path = Path(validation_result["artifact_path"])
else:
    _split_manifest_path = Path(
        "/kaggle/working/celebahq-validation-split-v2-81dcda99aeb9.json"
    )

preflight_result = run_celebahq_training_preflight(
    dire_root=Path("/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire"),
    stage2_artifact_dir=Path(
        "/kaggle/input/datasets/j4cknguyn/dire-celeba-hq-stage-2-artifacts-v1"
    ),
    split_manifest_path=_split_manifest_path,
    working_dir=Path("/kaggle/working"),
    artifact_dir=Path("/kaggle/working/celebahq-run-v2/artifacts"),
    checkpoint_dir=Path("/kaggle/working/celebahq-run-v2/checkpoints"),
)
print("Preflight status:", preflight_result["status"])
print("Input checks:", preflight_result["input_checks"])
print("Split check:", preflight_result["split_check"])
print("Disk:", preflight_result["disk"])
print("Write probes:", preflight_result["write_probes"])
print("GPU query:", preflight_result["gpu_query"])
print("Report:", preflight_result["report_path"])
print("Execution flags:", preflight_result["execution_flags"])
